# 01 資料初步稽核（Data Audit）

本 Notebook 為 NYCU R 語言期末專題之第一份分析筆記，目的在於對 SPARCS 2016 原始資料進行初步稽核，確認資料可順利讀取、欄位結構正確，並初步檢查資料型態與缺失值分布。

本研究重現 Jain et al. (2024) 使用 New York State SPARCS 2016 住院資料集預測住院天數（Length of Stay, LoS）之研究流程。由於原始 CSV 檔案約 1.05 GB，第一階段先讀取前 1,000 筆資料作為測試樣本，確認資料結構後，再於 `02_Preprocessing.ipynb` 讀取完整資料並進行正式前處理。

## 本章流程

1. 載入套件與設定資料路徑  
2. 讀取測試樣本資料  
3. 檢查資料維度與欄位名稱  
4. 檢查資料型態  
5. 檢查缺失值  
6. 檢視資料摘要  
7. 初步資料品質評估  
8. 初步結論


## 1.1 載入套件與設定資料路徑

本節載入資料讀取、資料整理與資料摘要所需之 R 套件，並設定 SPARCS 2016 原始 CSV 檔案路徑。

In [ ]:
#--------------------------------------------------
# 1.1 載入套件與設定資料路徑
#--------------------------------------------------

library(data.table)
library(dplyr)
library(tidyr)
library(stringr)
library(janitor)
library(skimr)

raw_file <- "../data/raw/Hospital_Inpatient_Discharges_(SPARCS_De-Identified)__2016_20260630.csv"

file.exists(raw_file)

## 1.2 讀取測試樣本資料

由於原始資料檔案較大，本 Notebook 先讀取前 1,000 筆資料進行資料稽核。此步驟目的不是進行正式分析，而是確認資料可正確載入，並檢查欄位結構是否符合後續分析需求。

In [ ]:
#--------------------------------------------------
# 1.2 讀取測試樣本資料
#--------------------------------------------------

df_sample <- fread(raw_file, nrows = 1000)

cat("====================================\n")
cat("NYCU R Final Project\n")
cat("測試樣本資料已成功載入\n")
cat("====================================\n")

dim(df_sample)

## 1.3 檢查資料維度與欄位名稱

本節檢查測試樣本之資料筆數、欄位數與欄位名稱。根據原論文描述，SPARCS 2016 原始資料應包含 34 個欄位，因此本步驟可初步確認目前讀入資料之欄位結構是否一致。

In [ ]:
#--------------------------------------------------
# 1.3 檢查資料維度與欄位名稱
#--------------------------------------------------

dim(df_sample)

names(df_sample)

## 1.4 檢查資料型態

本節使用 `str()` 檢查各欄位資料型態，包含文字型、整數型與數值型欄位。此步驟有助於判斷後續前處理與建模時是否需要進行型態轉換或類別變項編碼。

In [ ]:
#--------------------------------------------------
# 1.4 檢查資料型態
#--------------------------------------------------

str(df_sample)

## 1.5 檢查缺失值

本節初步檢查測試樣本中各欄位缺失值數量。正式缺失值分析將於 `02_Preprocessing.ipynb` 使用完整資料集進行。

In [ ]:
#--------------------------------------------------
# 1.5 檢查缺失值
#--------------------------------------------------

missing_summary_sample <- data.frame(
  Variable = names(df_sample),
  Missing = colSums(is.na(df_sample))
) %>%
  mutate(
    Percent = round(Missing / nrow(df_sample) * 100, 3)
  ) %>%
  arrange(desc(Missing))

missing_summary_sample

## 1.6 檢視資料摘要

本節使用 `skimr::skim()` 快速檢視測試樣本之整體摘要，包括各欄位資料型態、缺失值比例、數值變項分布與文字變項基本資訊。

In [ ]:
#--------------------------------------------------
# 1.6 檢視資料摘要
#--------------------------------------------------

skim(df_sample)

## 1.7 初步資料品質評估

測試樣本稽核結果顯示，SPARCS 2016 原始資料可成功匯入 R 環境，且測試樣本包含 34 個欄位，與原論文所描述之資料結構一致。

初步檢查結果包括：

- 原始 CSV 檔案可正確讀取。
- 測試樣本欄位數為 34 個。
- 欄位內容涵蓋醫院資訊、病人基本資料、住院資訊、診斷與處置分類、付款類型、出生體重及費用欄位。
- 資料同時包含類別型與數值型變項。
- 測試樣本缺失值情形有限；完整缺失值分布需於後續 Notebook 使用完整資料集確認。

此結果表示資料結構符合後續進行資料前處理與模型重現的基本需求。

## 1.8 初步結論

本 Notebook 完成 SPARCS 2016 原始資料之初步稽核。結果確認資料可成功載入，欄位數與研究需求一致，且資料型態與欄位內容符合後續分析需求。

下一份 Notebook `02_Preprocessing.ipynb` 將使用完整資料集進行正式資料前處理，包括缺失值處理、未知入院型態排除、資料洩漏變項移除、冗餘欄位刪除，以及 newborn 與 non-newborn 資料分割。